# Cybersecurity Incident Analytics — EDA
Production-ready exploratory analysis.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'src').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.config import CLEAN_DATA_FILE, EXPECTED_COLUMNS, NUMERIC_COLUMNS, CATEGORICAL_COLUMNS, BOOLEAN_COLUMNS
if not CLEAN_DATA_FILE.exists(): raise FileNotFoundError(f'Clean dataset not found: {CLEAN_DATA_FILE}')
df = pd.read_csv(CLEAN_DATA_FILE)
missing = [c for c in EXPECTED_COLUMNS if c not in df.columns]
if missing: raise ValueError(f'Missing required columns: {missing}')
if df.empty: raise ValueError('EDA dataset is empty.')
if df['incident_id'].isna().any() or df['incident_id'].duplicated().any(): raise ValueError('Missing or duplicate incident IDs.')
df['incident_date'] = pd.to_datetime(df['incident_date'], errors='coerce')
if df['incident_date'].isna().any(): raise ValueError('Invalid or missing incident dates.')
print(f'Rows: {len(df):,}')
print(f'Columns: {len(df.columns)}')

## Data quality

In [ ]:
quality = pd.DataFrame({'dtype': df.dtypes.astype(str), 'missing': df.isna().sum(), 'missing_pct': (df.isna().mean() * 100).round(2), 'unique': df.nunique(dropna=False)})
display(quality.sort_values('missing', ascending=False))

## Numeric and categorical analysis

In [ ]:
available_numeric = [c for c in NUMERIC_COLUMNS if c in df.columns]
display(df[available_numeric].describe().T.round(2))
for column in [c for c in CATEGORICAL_COLUMNS if c in df.columns]:
    print(f'\n{column}')
    display(df[column].value_counts(dropna=False).head(10).to_frame('count'))

## Incident trends and severity

In [ ]:
monthly = df.set_index('incident_date').resample('ME').size().rename('incident_count')
ax = monthly.plot(kind='line', figsize=(10, 4), marker='o')
ax.set_title('Monthly Cybersecurity Incident Volume')
ax.set_xlabel('Month')
ax.set_ylabel('Incidents')
plt.tight_layout()
plt.show()
ax = df['severity_score'].plot(kind='hist', bins=10, figsize=(8, 4))
ax.set_title('Severity Score Distribution')
plt.tight_layout()
plt.show()

## Financial and security indicators

In [ ]:
financial_columns = [c for c in ['ransom_demand_usd', 'regulatory_fine_usd'] if c in df.columns]
display(df[financial_columns].sum().to_frame('total_usd'))
boolean_rates = {c: round(float(df[c].mean() * 100), 2) for c in BOOLEAN_COLUMNS if c in df.columns}
display(pd.Series(boolean_rates, name='true_rate_percent').to_frame())
display(df[available_numeric].corr(numeric_only=True).round(2))

## Conclusion
Descriptive analysis only. Train/test-dependent transformations must be fitted on the training split in the future ML stage to avoid leakage.